# Lakebase Write Path Validation — Fraud Cases

Validates the native Postgres write path in `fe_bar_db.cases` (schema `cases`):

* **Tables:** `fraud_cases` (case lifecycle) and `case_actions` (audit log)
* **Module:** `lakebase_cases.py` — imported from the repo folder
* **Constraints:** partial unique index on `(txn_id) WHERE status <> 'closed'` prevents duplicate open cases; CHECK constraints on `status`, `assigned_queue`, `recommended_action`, `action_type`

### GRANT statements for the agent service principal (not run here)

```sql
-- Replace <service_principal> with the agent's SP name
GRANT USAGE ON SCHEMA cases TO <service_principal>;
GRANT SELECT, INSERT, UPDATE ON cases.fraud_cases TO <service_principal>;
GRANT SELECT, INSERT, UPDATE ON cases.case_actions TO <service_principal>;
GRANT USAGE ON SCHEMA fraud TO <service_principal>;
GRANT SELECT ON ALL TABLES IN SCHEMA fraud TO <service_principal>;
```

In [0]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "psycopg[binary]>=3.1.0", "databricks-sdk>=0.118.0"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import importlib, os
# Notebook lives in validation_notebooks/; repo modules are one level up
sys.path.insert(0, os.path.dirname(os.getcwd()))
import lakebase_cases
importlib.reload(lakebase_cases)
from lakebase_cases import LakebaseCases

print("Module loaded.")

In [0]:
lc = LakebaseCases()
conn = lc._get_conn()
with conn.cursor() as cur:
    cur.execute("""
        SELECT txn_id, customer_id, amount, amt_to_monthly_spend,
               is_cross_border, geo_distance_from_home, device_id, channel,
               risk_tier, risk_segment, is_fraud
        FROM fraud.txn_features
        WHERE is_cross_border = 1 AND amt_to_monthly_spend IS NOT NULL
        ORDER BY amt_to_monthly_spend DESC NULLS LAST
        LIMIT 1
    """)
    cols = [d.name for d in cur.description]
    row = cur.fetchone()

target_txn_id = row[0]
print("Highest-risk seeded cross-border transaction:")
for c, v in zip(cols, row):
    print(f"  {c}: {v}")

In [0]:
case_id = lc.create_fraud_case(
    txn_id=target_txn_id,
    recommended_action="block",
    assigned_queue="auto_block",
    notes=f"Highest amt_to_monthly_spend ({row[3]}x) for cross-border txn. "
          f"Geo distance {row[5]} km. Device {row[6]}, channel {row[7]}.",
    created_by="validation_test",
)
print(f"Case created: {case_id}")
print(f"Transaction: {target_txn_id}")

In [0]:
lc.close()

lc2 = LakebaseCases()
case = lc2.get_case(case_id)
print("Case read back on NEW connection:")
for k, v in case.items():
    if k != "actions":
        print(f"  {k}: {v}")
print(f"  actions: {len(case['actions'])} action(s)")
for a in case["actions"]:
    print(f"    [{a['action_type']}] {a['outcome']} by {a['actor']} at {a['created_at']}")

In [0]:
lc2.add_case_action(case_id, "recommendation", "AI model score 0.94 — recommend block", "validation_test")
lc2.add_case_action(case_id, "analyst_decision", "Confirmed fraudulent after review", "validation_test")

case = lc2.get_case(case_id)
print(f"Audit trail for case {case_id} ({len(case['actions'])} actions):")
for a in case["actions"]:
    print(f"  {a['created_at']} [{a['action_type']}] {a['outcome']} (by {a['actor']})")

In [0]:
case_id_2 = lc2.create_fraud_case(
    txn_id=target_txn_id,
    recommended_action="block",
    assigned_queue="auto_block",
    notes="Duplicate attempt",
    created_by="validation_test",
)
print(f"Original case_id: {case_id}")
print(f"Second call id:   {case_id_2}")
print(f"Same? {case_id == case_id_2}")
assert case_id == case_id_2, "Idempotency failed!"

In [0]:
def expect_error(label, fn):
    try:
        fn()
        print(f"  ✗ {label}: NO ERROR (expected one)")
    except Exception as e:
        print(f"  ✓ {label}: {type(e).__name__}: {str(e)[:200]}")
    finally:
        # Always rollback to clear any failed transaction state
        try:
            lc2._get_conn().rollback()
        except:
            lc2._conn = None  # force reconnect

print("Rejection tests:")

# 1. Invalid recommended_action (use a valid txn_id)
expect_error("Invalid recommended_action", lambda: lc2.create_fraud_case(
    txn_id=target_txn_id, recommended_action="INVALID",
    assigned_queue="auto_block", notes=None, created_by="validation_test"))

# 2. Invalid status
expect_error("Invalid status", lambda: lc2.update_case_status(case_id, "INVALID", "validation_test"))

# 3. txn_id not in fraud.txn_features
expect_error("Non-existent txn_id", lambda: lc2.create_fraud_case(
    txn_id="TXN-DOES-NOT-EXIST", recommended_action="block",
    assigned_queue="auto_block", notes=None, created_by="validation_test"))

In [0]:
lc2.update_case_status(case_id, "closed", "validation_test")

case = lc2.get_case(case_id)
print(f"Final case state:")
for k, v in case.items():
    if k != "actions":
        print(f"  {k}: {v}")
print(f"\nFull audit trail ({len(case['actions'])} actions):")
for a in case["actions"]:
    print(f"  {a['created_at']} [{a['action_type']}] {a['outcome']}")

In [0]:
import time

# Get 50 txn_ids for benchmarking
conn_bench = lc2._get_conn()
with conn_bench.cursor() as cur:
    cur.execute("SELECT txn_id FROM fraud.txn_features ORDER BY random() LIMIT 51")
    bench_txns = [r[0] for r in cur.fetchall()]

# Warm-up
warmup_id = lc2.create_fraud_case(
    txn_id=bench_txns[0], recommended_action="review",
    assigned_queue="analyst_review", notes="warmup", created_by="validation_test")
_ = lc2.get_case(warmup_id)
lc2.update_case_status(warmup_id, "closed", "validation_test")

latencies = []
for txn_id in bench_txns[1:]:
    t0 = time.perf_counter()
    cid = lc2.create_fraud_case(
        txn_id=txn_id, recommended_action="review",
        assigned_queue="analyst_review", notes="latency test", created_by="validation_test")
    _ = lc2.get_case(cid)
    latencies.append((time.perf_counter() - t0) * 1000)
    # Close case so partial unique index doesn't block next run
    lc2.update_case_status(cid, "closed", "validation_test")

latencies.sort()
p50 = latencies[len(latencies) // 2]
p95 = latencies[int(len(latencies) * 0.95)]
print(f"Write latency (50 create-and-read-back round trips, reused connection):")
print(f"  p50: {p50:.2f} ms")
print(f"  p95: {p95:.2f} ms")
print(f"  min: {min(latencies):.2f} ms, max: {max(latencies):.2f} ms")

In [0]:
print("Querying fe_bar_lakebase.cases.fraud_cases via Unity Catalog...")
try:
    uc_df = spark.sql("SELECT case_id, txn_id, status, created_by, created_at FROM fe_bar_lakebase.cases.fraud_cases LIMIT 10")
    display(uc_df)
    print(f"\n✓ UC visibility confirmed: {uc_df.count()} rows returned")
except Exception as e:
    print(f"UC query not supported from serverless notebook compute: {e}")
    print("\nFalling back to Postgres direct query:")
    conn_uc = lc2._get_conn()
    with conn_uc.cursor() as cur:
        cur.execute("SELECT case_id, txn_id, status, created_by, created_at FROM cases.fraud_cases ORDER BY created_at DESC LIMIT 10")
        cols = [d.name for d in cur.description]
        rows = cur.fetchall()
    print(f"{'case_id':<40} {'txn_id':<16} {'status':<12} {'created_by':<20} {'created_at'}")
    print("-" * 120)
    for r in rows:
        print(f"{str(r[0]):<40} {r[1]:<16} {r[2]:<12} {r[3]:<20} {r[4]}")

lc2.close()